# Adaptacyjne planowanie 

## Cel: Nauczysz się projektować systemy, które potrafią zmienić zdanie w trakcie działania, zamiast ślepo podążać za nieaktualnym planem.

> "Plans are nothing; planning is everything."
>
> "Plany są niczym, planowanie jest wszystkim."
> — Dwight D. Eisenhower


### Czego się nauczysz:
1. czym różni się podejście statyczne od adaptacyjnego w systemach AI
2. jak działa pętla *ReAct* w kontekście dużych modeli językowych
3. dlaczego warto stosować *Schema-Guided Reasoning* do weryfikacji decyzji
4. jak zarządzać opóźnieniami wynikającymi z procesu myślowego agenta
5. jak zdefiniować sztywną przestrzeń akcji przy użyciu Pydantic Literal
6. w jaki sposób podzielić pamięć agenta na warstwy konstytucyjne i epizodyczne
7. jak zaimplementować lejek decyzyjny ograniczający halucynacje modelu


Wchodzimy w kluczowy mechanizm tej lekcji: logikę `NextStep`.

Twoim zadaniem na teraz jest zrozumienie, jak spiąć system od początku do końca (end-to-end). Zamiast skupiać się na izolowanych fragmentach kodu, zidentyfikujemy i połączymy wszystkie niezbędne elementy: od struktury danych, przez pętlę decyzyjną, aż po wykonanie akcji. Zrozumienie tej mechaniki teraz jest krytyczne – dzięki temu w kolejnych lekcjach nie będziemy tracić czasu na podstawy, tylko od razu przejdziemy do zaawansowanej dynamiki.





### Plan gry na tę lekcję

Wiem, że w tej lekcji pojawi się sporo teorii i kodu, ale spokojnie. Całą logikę opieramy na kilku kluczowych elementach, które tworzą **cykliczny** (nie liniowy!) przepływ decyzyjny:

1. `ReservationGoal` – **Cel**. Definiujemy, co chcemy osiągnąć (rezerwacja stolika z określonymi parametrami).
2. `NextStep` – **Decyzja**. LLM analizuje sytuację, tworzy plan i wybiera **tylko jeden** następny krok.
3. `dispatch()` – **Wykonanie**. Tłumaczymy strukturę danych na naturalną wypowiedź.
4. `run_reservation_agent()` – **Pętla główna**. Spinamy wszystko w cykl, który powtarza się aż do sukcesu lub porażki.
5. **Pamięć wielowarstwowa** – Constitutional, Semantic, Episodic, Working Memory.

Spójrz na diagram przepływu, który będziemy realizować:

```text
┌─────────────────────────────────────────────────────────────────────────────────────┐
│                     🍽️  ADAPTIVE RESERVATION AGENT FLOW                             │
│                              (ReAct Pattern)                                        │
└─────────────────────────────────────────────────────────────────────────────────────┘

     ┌───────────────────┐
     │  👤 KLIENT        │
     │  "Zarezerwuj      │
     │   stolik..."      │
     └─────────┬─────────┘
               │
               ▼
╔══════════════════════════════════════════════════════════════════════════════════════╗
║  1️⃣  GOAL DEFINITION                 ReservationGoal                                 ║
╠══════════════════════════════════════════════════════════════════════════════════════╣
║                                                                                      ║
║    ┌─────────────────────────────────────────────────────────────────────────────┐   ║
║    │  📋 ReservationGoal                                                         │   ║
║    │  • restaurant_name: str         ("Bella Italia")                            │   ║
║    │  • date: str                    ("2024-01-20")                              │   ║
║    │  • time: str                    ("19:00")                                   │   ║
║    │  • party_size: int              (2)                                         │   ║
║    │  • customer_name: str           ("Kowalski")                                │   ║
║    │  • time_flexibility_minutes: int (30) ← granica autonomii agenta            │   ║
║    └─────────────────────────────────────────────────────────────────────────────┘   ║
║                                                                                      ║
╚══════════════════════════════════════════════════════════════════════════════════════╝
               │
               ▼
╔══════════════════════════════════════════════════════════════════════════════════════╗
║  2️⃣  ReAct LOOP                        run_reservation_agent()                       ║
╠══════════════════════════════════════════════════════════════════════════════════════╣
║                                                                                      ║
║    ┌─────────────────────────────────────────────────────────────────────────────┐   ║
║    │                                                                             │   ║
║    │      ┌───────────────┐      ┌───────────────┐      ┌───────────────┐        │   ║
║    │      │   🧠 THOUGHT  │      │   ⚡ ACTION    │      │ 👁️ OBSERVATION│        │   ║
║    │      │   (Reasoning) │─────▶│   (Acting)    │─────▶│   (Feedback)  │        │   ║
║    │      └───────────────┘      └───────────────┘      └───────┬───────┘        │   ║
║    │              ▲                                             │                │   ║
║    │              │                                             │                │   ║
║    │              └─────────────────────────────────────────────┘                │   ║
║    │                           🔄 PĘTLA ReAct                                    │   ║
║    └─────────────────────────────────────────────────────────────────────────────┘   ║
║                                                                                      ║
║    ┌─────────────────────────────────────────────────────────────────────────────┐   ║
║    │                         📤 NextStep (LLM Output)                            │   ║
║    │  ┌───────────────────────────────────────────────────────────────────────┐  │   ║
║    │  │                                                                       │  │   ║
║    │  │  🧠 THOUGHT (Reasoning)                                               │  │   ║
║    │  │  ├─ current_state: str        "Restauracja zaproponowała 19:30,       │  │   ║
║    │  │  │                             to mieści się w elastyczności ±30min"  │  │   ║
║    │  │  │                                                                    │  │   ║
║    │  │  └─ plan_remaining_steps:     ["zaakceptować 19:30",                  │  │   ║
║    │  │                                "podać nazwisko",                      │  │   ║
║    │  │                                "potwierdzić rezerwację"]              │  │   ║
║    │  │                                                                       │  │   ║
║    │  │  ⚡ ACTION (Acting)                                                    │  │   ║
║    │  │  └─ next_action: Accepting    ─────────────────────────────────────┐  │  │   ║
║    │  │                                                                    │  │  │   ║
║    │  └────────────────────────────────────────────────────────────────────│──┘  │   ║
║    └───────────────────────────────────────────────────────────────────────│─────┘   ║
║                                                                            │         ║
║    ⚠️ BEZPIECZNIK: max 10 iteracji (ochrona przed nieskończoną pętlą)      │         ║
║                                                                            │         ║
╚════════════════════════════════════════════════════════════════════════════│═════════╝
                                                                             │
            ┌────────────────────────────────────────────────────────────────┘
            │
            ▼
╔══════════════════════════════════════════════════════════════════════════════════════╗
║  3️⃣  ACTION SPACE (7 KART)              next_action wybiera JEDNĄ                    ║
╠══════════════════════════════════════════════════════════════════════════════════════╣
║                                                                                      ║
║    Agent ma "talię kart" – nie może wymyślać akcji, tylko wybierać z zestawu:        ║
║                                                                                      ║
║    ┌────────────┐ ┌────────────┐ ┌────────────┐ ┌────────────┐ ┌─────────────┐       ║
║    │1. GREETING │ │ 2. ASKING  │ │3. ACCEPTING│ │4.COUNTERING│ │5. PROVIDING │       ║
║    │            │ │            │ │            │ │            │ │   DETAILS   │       ║
║    │ "Dzień     │ │ "Czy macie │ │ "Tak,      │ │ "A może    │ │             │       ║
║    │  dobry..." │ │  stolik?"  │ │  pasuje"   │ │  19:30?"   │ │ "Kowalski,  │       ║
║    │            │ │            │ │            │ │            │ │  2 osoby"   │       ║
║    └────────────┘ └────────────┘ └────────────┘ └────────────┘ └─────────────┘       ║
║                                                                                      ║
║                         ┌────────────────┐    ┌────────────────┐                     ║
║                         │ 6. CONFIRMING  │    │  7. FAILING    │                     ║
║                         │                │    │                │                     ║
║                         │ "Dziękuję,     │    │ "Rozumiem,     │                     ║
║                         │  do widzenia"  │    │  szkoda..."    │                     ║
║                         │                │    │                │                     ║
║                         │   ✅ SUKCES    │    │   ❌ PORAŻKA   │                     ║
║                         │   (TERMINAL)   │    │   (TERMINAL)   │                     ║
║                         └────────────────┘    └────────────────┘                     ║
║                                                                                      ║
╚══════════════════════════════════════════════════════════════════════════════════════╝
                                       │
                                       ▼
╔══════════════════════════════════════════════════════════════════════════════════════╗
║  4️⃣  EXECUTION                          dispatch()                                   ║
╠══════════════════════════════════════════════════════════════════════════════════════╣
║                                                                                      ║
║    Naturalna wypowiedź                                                               ║
║                                                                                      ║
║    ┌─────────────────────────────────────────────────────────────────────────────┐   ║
║    │                                                                             │   ║
║    │   Asking(date="2024-01-20", time="19:00", party_size=2)                     │   ║
║    │                              │                                              │   ║
║    │                              ▼                                              │   ║
║    │                        dispatch()                                           │   ║
║    │                              │                                              │   ║
║    │                              ▼                                              │   ║
║    │   "Czy mają Państwo wolny stolik na 20 stycznia o godzinie 19:00            │   ║
║    │    dla 2 osób?"                                                             │   ║
║    │                                                                             │   ║
║    └─────────────────────────────────────────────────────────────────────────────┘   ║
║                                                                                      ║
║    ⚠️ Tu też można dodać GUARDRAILS (walidacja dat, limitów osób itd.)               ║
║                                                                                      ║
╚══════════════════════════════════════════════════════════════════════════════════════╝
                                       │
                                       ▼
╔══════════════════════════════════════════════════════════════════════════════════════╗
║  5️⃣  OBSERVATION                        Feedback z zewnątrz                          ║
╠══════════════════════════════════════════════════════════════════════════════════════╣
║                                                                                      ║
║    ┌─────────────────────────────────────────────────────────────────────────────┐   ║
║    │                                                                             │   ║
║    │                          🏪 RESTAURACJA                                     │   ║
║    │                          (człowiek / system)                                │   ║
║    │                                                                             │   ║
║    │                                 │                                           │   ║
║    │                                 ▼                                           │   ║
║    │                                                                             │   ║
║    │         "Niestety o 19:00 jest zajęte. Mogę zaproponować 19:30"             │   ║
║    │                                                                             │   ║
║    └─────────────────────────────────────────────────────────────────────────────┘   ║
║                                       │                                              ║
║                                       │  OBSERVATION wraca do pętli ReAct            ║
║                                       │  (dodane do messages → THOUGHT)              ║
║                                       │                                              ║
║                                       ▼                                              ║
║                        ┌──────────────────────────────┐                              ║
║                        │  🔄 POWRÓT DO THOUGHT        │                              ║
║                        │                              │                              ║
║                        │  Agent REPLANUJE:            │                              ║
║                        │  "19:30 mieści się w ±30min, │                              ║
║                        │   mogę zaakceptować"         │                              ║
║                        │                              │                              ║
║                        │  📍 TO JEST ADAPTACJA!       │                              ║
║                        └──────────────────────────────┘                              ║
║                                                                                      ║
╚══════════════════════════════════════════════════════════════════════════════════════╝
                                       │
                                       ▼
╔══════════════════════════════════════════════════════════════════════════════════════╗
║  6️⃣  MEMORY LAYERS                     Pamięć wielowarstwowa                         ║
╠══════════════════════════════════════════════════════════════════════════════════════╣
║                                                                                      ║
║    Agent korzysta z 4 warstw pamięci (od najbardziej stałej do dynamicznej):         ║
║                                                                                      ║
║    ┌─────────────────────────────────────────────────────────────────────────────┐   ║
║    │  🏛️ CONSTITUTIONAL MEMORY          (NIEZMIENNE - "DNA agenta")              │   ║
║    │  ─────────────────────────────────────────────────────────────────────────  │   ║
║    │  • max_time_flexibility: 60 min   (absolutny limit, nawet jeśli klient      │   ║
║    │  • max_counter_proposals: 2         chce więcej)                            │   ║
║    │  • always_mention_allergies: True                                           │   ║
║    │  • require_confirmation: True                                               │   ║
║    ├─────────────────────────────────────────────────────────────────────────────┤   ║
║    │  🧠 SEMANTIC MEMORY                (PROFIL KLIENTA - trwała wiedza)         │   ║
║    │  ─────────────────────────────────────────────────────────────────────────  │   ║
║    │  • name: "Jan Kowalski"                                                     │   ║
║    │  • vip_level: "gold"                                                        │   ║
║    │  • allergies: ["orzechy"]          ← Agent MUSI o tym powiedzieć!           │   ║
║    │  • preferences: ["stolik przy oknie", "cicha okolica"]                      │   ║
║    ├─────────────────────────────────────────────────────────────────────────────┤   ║
║    │  📚 EPISODIC MEMORY                (HISTORIA - uczenie się na błędach)      │   ║
║    │  ─────────────────────────────────────────────────────────────────────────  │   ║
║    │  • 2024-01-15 @ Bella Italia: ✅ sukces, "stolik przy oknie"                │   ║
║    │  • 2024-02-20 @ Pod Różą: ⚠️ "PROBLEM: czekał 30 min mimo rezerwacji"       │   ║
║    │                                    ↳ Agent może być bardziej asertywny      │   ║
║    ├─────────────────────────────────────────────────────────────────────────────┤   ║
║    │  ⚡ WORKING MEMORY                 (BIEŻĄCA SESJA - RAM)                     │   ║
║    │  ─────────────────────────────────────────────────────────────────────────  │   ║
║    │  • messages: [...]                 (historia rozmowy)                       │   ║
║    │  • counter_proposals_count: 1      (ile razy już negocjowaliśmy)            │   ║
║    │  • previous_plan: ["asking",       (plan z poprzedniej iteracji -           │   ║
║    │                    "providing..."]   pozwala wykryć zmianę strategii)       │   ║
║    └─────────────────────────────────────────────────────────────────────────────┘   ║
║                                                                                      ║
╚══════════════════════════════════════════════════════════════════════════════════════╝
                                       │
                                       ▼
╔══════════════════════════════════════════════════════════════════════════════════════╗
║  7️⃣  TERMINAL STATES                   Zakończenie pętli ReAct                       ║
╠══════════════════════════════════════════════════════════════════════════════════════╣
║                                                                                      ║
║         ┌─────────────────────────────┐    ┌─────────────────────────────┐           ║
║         │                             │    │                             │           ║
║         │      ✅ CONFIRMING          │    │      ❌ FAILING             │           ║
║         │         (SUKCES)            │    │        (PORAŻKA)            │           ║
║         │                             │    │                             │           ║
║         │  "Stolik zarezerwowany      │    │  "Nie udało się znaleźć     │           ║
║         │   na 19:30, na nazwisko     │    │   terminu w akceptowalnym   │           ║
║         │   Kowalski, 2 osoby"        │    │   zakresie czasowym"        │           ║
║         │                             │    │                             │           ║
║         │  📊 Raport dla klienta:     │    │  📊 Raport dla klienta:     │           ║
║         │  • Gdzie: Bella Italia      │    │  • Przyczyna: brak miejsc   │           ║
║         │  • Kiedy: 20.01, 19:30      │    │  • Próby: 3 negocjacje      │           ║
║         │  • Uwagi: alergia zgłoszona │    │  • Sugestia: inna restaur.  │           ║
║         │                             │    │                             │           ║
║         └─────────────────────────────┘    └─────────────────────────────┘           ║
║                                                                                      ║
╚══════════════════════════════════════════════════════════════════════════════════════╝
```

Mówiąc krótko, **plan na dzisiaj** (pełny flow Adaptive Reservation Agent):

| Etap | Funkcja/Struktura | Co robi? |
|------|-------------------|----------|
| 1. Cel | `ReservationGoal` | Definiuje parametry rezerwacji + granice autonomii |
| 2. Decyzja | `NextStep` + LLM | Analizuje, planuje, wybiera **jedną** akcję |
| 3. Akcje | 7 klas Pydantic | Sztywna przestrzeń możliwych ruchów (karty) |
| 4. Wykonanie | `dispatch()` | Tłumaczy obiekt → naturalną wypowiedź |
| 5. Pętla | `run_reservation_agent()` | Spina wszystko w cykl *ReAct* |


Na koniec jeszcze zadanie domowe i dotkniemy warstwy pamięci: `Constitutional → Semantic → Episodic → Working`.

## Dobór narzędzia do problemu

W DataWorkshop uczymy poziomów od 2 do 5+. Ważne, aby pamiętać, że każdy z tych poziomów jest "production-ready". To nie jest tak, że poziom 5 jest "lepszy" od 2. Po prostu każdy poziom ma inne ograniczenia i służy do rozwiązywania innej klasy problemów. Wybór zależy od tego, na ile autonomii (kontra kontroli człowieka) pozwala Twój konkretny use case.

- **Poziom 2 (Programowalne AI)** świetnie radzi sobie w przewidywalnym środowisku. Jeśli masz zadanie typu "Pobierz fakturę -> Wyślij maila", sztywny scenariusz jest idealny. Jest szybki i tani.
- **Poziom 5 (Strategiczne AI)** wchodzi do gry tam, gdzie pojawia się większa **niepewność**.

### Problem "Listy zakupów" w dynamicznym świecie

Wielu inżynierów próbuje rozwiązywać skomplikowane problemy metodami z Poziomu 2. Proszą model o wygenerowanie statycznej listy kroków (`plans: List[str]`) na samym początku:

```json
{
  "plan": [
    "Wejdź na stronę X",
    "Kliknij przycisk Y",
    "Pobierz dane",
    "Zapisz raport"
  ]
}
```

To działa jak lista zakupów. Ale rzeczywistość rzadko jest tak liniowa.

Wyobraź sobie prostą, ludzką sytuację: biegniesz na autobus. Twój "plan" to: dobiec do drzwi i wsiąść. Ale w połowie drogi widzisz, że kierowca zamknął drzwi i odjeżdża.
Co robi człowiek? Nie biegnie dalej, by uderzyć nosem w odjeżdżający pojazd (realizacja statycznego planu). Człowiek **przerywa** działanie, ocenia nową sytuację i zmienia plan (np. "zamawiam taksówkę" albo "czekam na następny").

To jest właśnie esencja **Poziomu 5**. To zdolność do powiedzenia: *"Ups, warunki się zmieniły, to, co planowałem (sekundę temu), nie ma już sensu"*.

### Analogia szachowa

Najlepszym przykładem problemu, którego nie da się rozwiązać na Poziomie 2, są szachy.

Gdybyśmy chcieli grać w szachy modelem statycznym (Poziom 2), musielibyśmy przed rozpoczęciem partii zaplanować 10 ruchów do przodu i zapisać je na kartce. *"Pionek E4, Skoczek F3..."*. Zaczynamy grać. Wykonujemy ruchy w ciemno, ignorując to, co robi przeciwnik. Efekt? Przeciwnik robi coś niespodziewanego w drugim ruchu, a nasz misterny plan prowadzi do katastrofy.

Szachy wymagają podejścia **adaptacyjnego** (Poziom 5). Masz strategię, ale decyzję o konkretnym ruchu podejmujesz dopiero po ocenie nowej sytuacji na planszy.



### AI Coding (Cursor, Claude, OpenAI)

Przenieśmy to na grunt, który możesz kojarzyć. Obecnie popularne staje się pisanie Software 1.0 (klasycznego kodu) przy użyciu AI – narzędzi takich jak Cursor, Claude Code czy Gemini CLI.

Wyobraź sobie, że prosisz model o napisanie mikroserwisu w FastAPI.

- Podejście statyczne (Poziom 2): Model generuje piękny plan: "1. Stwórz `models.py`, 2. Stwórz `main.py`, 3. Napisz testy". Następnie generuje kod dla wszystkich trzech plików naraz. Ty próbujesz to uruchomić i... `ImportError`. Okazuje się, że w kroku 2 użył biblioteki, której zapomniał dodać w kroku 1. Plan był, ale był sztywny. Model nie sprawdził, czy krok 1 zadziałał, zanim zrobił krok 2.
- Podejście adaptacyjne (Poziom 5): Model też zaczyna od planu. Ale po wygenerowaniu `models.py` próbuje go uruchomić (lub zwalidować). Jeśli napotka błąd, **zatrzymuje się**. Mówi: *"Planowałem teraz pisać `main.py`, ale widzę, że baza danych nie wstaje. Muszę zmodyfikować plan: najpierw naprawię `docker-compose.yml`, a dopiero potem wrócę do kodu aplikacji"*.

To jest kluczowa różnica, którą musisz zapamiętać: planowanie jest dostępne już na poziomie 2. Ale korekta planu (adaptacja) pojawia się dopiero na Poziomie 5. Właśnie dlatego w tym module budujemy system, który nie tylko "ma plan", ale potrafi go bez żalu wyrzucić do kosza, gdy rzeczywistość (np. błąd kompilacji) tego wymaga.


W Software 3.0 oznacza to zmianę architektury z liniowej na cykliczną:
1.  Wykonaj akcję
2.  Zatrzymaj się i spójrz na wynik (Feedback)
3.  Zdecyduj co dalej (kontynuuj, zmień plan lub zakończ)

## ⚠️ Koszty autonomii

Wchodząc na Poziom 5, musisz zaakceptować, że budujesz inny rodzaj oprogramowania. To nie jest już prosta funkcja `input -> output`. To system, który zarządza większą złożonością.

### Cena inteligencji (*latency*)
Musimy to powiedzieć wprost: rozumowanie kosztuje (czas, tokeny, pieniądze).

System na Poziomie 5 wykonuje "pracę myślową" (reasoning tokens) oraz fizyczną (API calls). To trwa.
- Jeśli oczekujesz odpowiedzi w czasie rzeczywistym (200-500ms) – poziom 5 jest raczej złym wyborem.
- Współczesne modele "reasoning" potrzebują od kilku do kilkunastu sekund na sam proces myślowy. Do tego dochodzą narzuty sieciowe (latency narzędzi).

Mental Model: System 1 vs System 2:
Warto przypomnieć koncepcję z książki *"Pułapki myślenia"* Daniela Kahnemana:
- System 1 (szybki): Intuicyjny, automatyczny, natychmiastowy. To jest klasyczny Chatbot
- System 2 (wolny): Analityczny, wymagający wysiłku i czasu. To jest poziom 5 lub wyżej

W praktyce architektonicznej można łączyć oba te światy. Użytkownik rozmawia z szybkim "Systemem 1" (interfejs), podczas gdy w tle "System 2" (Agent) wykonuje ciężką pracę analityczną, korygując i uzupełniając odpowiedzi asynchronicznie.


### Dynamiczne planowanie

Fakt, że LLM jest niedeterministyczny, nie jest wadą, którą trzeba ukrywać. To cecha, którą trzeba umiejętnie zarządzać. Łatwo jest powiedzieć, że LLM halucynuje, dlatego popełnia błędy; trudniej jest zdobyć kompetencje, aby tym zarządzać.

Na poziomie 5 nie próbujemy wyeliminować błędów za wszelką cenę (bo to niemożliwe), ale budujemy mechanizm, który potrafi je wykryć i naprawić.

Proces wygląda następująco:

1.  Generujemy plan na N kroków do przodu. To nie musi być jeden krok. Możemy (i powinniśmy) patrzeć szerzej. Agent może zaplanować 3, 5, a nawet 10 kroków (np. "Wyszukaj X -> Pobierz PDF -> Przeanalizuj -> Wyślij maila"). To daje mu kontekst i cel.

2.  Wykonujemy **TYLKO** pierwszy krok. Mimo że mamy plan na całą podróż, fizycznie wykonujemy tylko najbliższą akcję. To jest nasz "commit".

3.  Analizujemy informację zwrotną. Patrzymy na wynik tej akcji. Co się stało? Czy dostaliśmy to, co chcieliśmy? A może pojawił się błąd?

4.  Generujemy plan od nowa. To jest moment adaptacji. Bierzemy pod uwagę historię (co już zrobiliśmy) i nowy stan wiedzy.
    - Jeśli wszystko poszło zgodnie z założeniami (Happy Path) -> nowy plan może być identyczny z poprzednim (tylko krótszy o jeden krok).
    - Jeśli coś poszło nie tak -> stary plan ląduje w koszu, a agent generuje zupełnie nową ścieżkę dotarcia do celu.

W tej architekturze **plan jest jednorazowy**. Służy tylko do podjęcia najlepszej decyzji "tu i teraz". To pozwala nam łączyć strategiczne myślenie (patrzenie w przód) z taktyczną elastycznością (reagowanie na błędy).

To podejście przypomina filozofię Jeffa Bezosa z Amazonu: jeśli czekasz na 100% pewności, działasz zbyt wolno. Wystarczy, że masz mechanizm szybkiej korekty. Nawet jeśli w pierwszej iteracji plan był błędny, w drugiej iteracji system go "naprawi", bo dostanie feedback z rzeczywistości.


## Rezerwacja stolika 

Pamiętasz lekcję 1.3? Robiliśmy tam prosty skrypt rezerwacyjny. Czy on "nie miał planu"? Miał. Ale ten plan był zakodowany na sztywno przez nas – w *system prompcie* lub w logice `if/else`.

W Module 2 poznaliśmy też *Workflow Memory* (Poziom 4). Tam plan był bardziej zaawansowany – system wiedział, w którym jest kroku, potrafił ukrywać wykonane zadania i skupiać się na obecnych. Ale uwaga: to nadal była ścieżka wytyczona przez inżyniera. Agent poruszał się wprost po szynach, które Ty położyłeś. Jeśli rozmowa zboczyła na tory, których nie przewidziałeś w definicji Workflow, system wypadał z rytmu.


Teraz zmieniamy reguły gry. Plan staje się bardziej elastyczny. To LLM decyduje, czy następnym krokiem jest "potwierdzenie danych", czy może nagły "zwrot akcji i negocjacja innej godziny".

Aby jednak ten proces nie zamienił się w chaos, musimy narzucić mu strukturę myślenia. Zastosujemy tu wzorce projektowe z lekcji 4.1 (Kaskada, Cykl, Routing).



Najpierw przypomnij, co chcemy osiągnąć.  Z lotu ptaka proces wygląda tak:


```
┌─────────────┐         ┌─────────────┐         ┌─────────────┐
│   KLIENT    │ zleca   │   AGENT AI  │ dzwoni  │ RESTAURACJA │
│             │ ──────► │             │ ──────► │ (człowiek)  │
│ "Zarezerwuj │         │ planuje,    │ ◄────── │ odpowiada   │
│  stolik..." │         │ rozmawia    │         │             │
└─────────────┘         └─────────────┘         └─────────────┘
```

Agent **nie zna** dostępności - musi zapytać i w razie czego negocjować.


```
┌─────────────────────────────────────────────────────────────────────────────┐
│                           SCHEMAT DZIAŁANIA                                 │
└─────────────────────────────────────────────────────────────────────────────┘

    KLIENT                      AGENT AI                         RESTAURACJA
    (zleca)                    (prowadzi)                        (odpowiada)
       │                           │                                  │
       │  ReservationGoal          │                                  │
       │  ─────────────────►       │                                  │
       │  • restaurant_name        │      "Dzień dobry..."            │
       │  • date: "Jutro"          │  ────────────────────────────►   │
       │  • time: "19:00"          │                                  │
       │  • party_size: 2          │      "Słucham?"                  │
       │  • customer_name          │  ◄────────────────────────────   │
       │                           │                                  │
       │                           │      "Czy mają Państwo..."       │
       │                           │  ────────────────────────────►   │
       │                           │                                  │
       │                           │      "O 19:00 zajęte, jest 19:30"│
       │                           │  ◄────────────────────────────   │
       │                           │                                  │
       │                           │   [Agent DECYDUJE: akceptować?]  │
       │                           │                                  │
       │                           │      "19:30 pasuje, Kowalski"    │
       │                           │  ────────────────────────────►   │
       │                           │                                  │
       │                           │      "Zarezerwowane!"            │
       │  ◄─────────────────       │  ◄────────────────────────────   │
       │  SUKCES: 19:30            │                                  │
       │                           │                                  │
       
```  

Przechodzimy od teorii szachowej do kodu.

## Wczytujemy biblioteki

In [1]:
from enum import Enum
from typing import List, Optional, Union, Literal, Annotated
from annotated_types import MinLen, MaxLen
from pydantic import BaseModel, Field
import helper as h

## Definicja Celu (input i logika)

Zaczynamy od zdefiniowania kontekstu misji – czyli tego, co dostajemy od klienta. W tej strukturze zamykamy zarówno twarde dane (data, godzina), jak i logikę biznesową (np. elastyczność czasowa czy limit prób), która wyznaczy bezpieczne ramy dla autonomicznych decyzji agenta bez zbędnego komplikowania modelu danych.


```text

┌─────────────────────────────────────────────────────────────────────────┐
│                           SYSTEM AGENTA                                 │
├─────────────────────────────────────────────────────────────────────────┤
│                                                                         │
│   ┌──────────────┐         ┌──────────────┐         ┌──────────────┐    │
│   │     GOAL     │         │     LLM      │         │   DISPATCH   │    │
│   │              │         │              │         │              │    │
│   │ "Co chcemy   │────────▶│ "Co zrobić   │────────▶│ "Jak to      │    │
│   │  osiągnąć?"  │         │  teraz?"     │         │  powiedzieć?"│    │
│   │              │         │              │         │              │    │
│   │ • restauracja│         │ • analizuje  │         │ • generuje   │    │
│   │ • data       │         │ • planuje    │         │   tekst      │    │
│   │ • godzina    │         │ • wybiera    │         │ • formatuje  │    │
│   │ • osoby      │         │   akcję      │         │   wypowiedź  │    │
│   └──────────────┘         └──────────────┘         └──────────────┘    │
│          │                        │                        │            │
│          │                        │                        │            │
│          ▼                        ▼                        ▼            │
│   ┌─────────────────────────────────────────────────────────────────┐   │
│   │                      PĘTLA GŁÓWNA                               │   │
│   │                   run_reservation_agent()                       │   │
│   │                                                                 │   │
│   │    [Start] ──▶ [LLM decyduje] ──▶ [Dispatch mówi] ──▶ [Koniec?] │   │
│   │                      ▲                                    │     │   │
│   │                      └────────── [Restauracja odpowiada] ◀┘     │   │
│   └─────────────────────────────────────────────────────────────────┘   │
│                                                                         │
└─────────────────────────────────────────────────────────────────────────┘

```

### `ReservationGoal`

In [2]:
class ReservationGoal(BaseModel):
    restaurant_name: str
    date: str
    preferred_time: str
    party_size: int
    customer_name: str
    time_flexibility_minutes: int = 30
    special_requests: Optional[str] = None

## Definiowanie stanów 

Choć wchodzimy na poziom 5, gdzie agent planuje, koreguje swoje plany oraz podejmowaniu decyzji, w kwestii **działania** nadal chcemy zachować kontrolę inżynierską. Bardzo ostrożnie przekazujemy mu władzę nad tym, co może zrobić.

Teoretycznie moglibyśmy pozwolić modelowi wymyślać własne akcje w locie (tzw. *Dynamic Action Space*), ale na tym etapie to zbyt ryzykowne. Dlatego stosujemy architekturę hybrydową:
1.  **Reasoning (myślenie):** Pełna elastyczność. Agent sam decyduje *dlaczego* i *kiedy* zmienić strategię.
2.  **Actions (stany):** Sztywne ramy. Agent może wybrać tylko jedną z z góry zdefiniowanych opcji.

Wyobraź to sobie jak grę w karty. Agent jest mistrzem strategii (może blefować, zmieniać taktykę, adaptować się do przeciwnika), ale talia kart jest z góry ustalona. Nie może wyciągnąć z rękawa karty, która nie istnieje.

```text
 ┌─────────────────────────────────────────────────────────────────────────────┐
 │                            DOSTĘPNE STANY (KARTY)                           │
 └─────────────────────────────────────────────────────────────────────────────┘

  ┌────────────────────────────────────────────────────────────────────────────┐
  │  #  │  STAN (AKCJA)      │  OPIS                     │  PRZYKŁAD           │
  ├─────┼────────────────────┼───────────────────────────┼─────────────────────┤
  │  1  │  Greeting          │  Rozpoczęcie rozmowy      │  "Dzień dobry..."   │
  │  2  │  Asking            │  Pytanie o dostępność     │  "Czy macie...?"    │
  │  3  │  Accepting         │  Akceptacja propozycji    │  "Tak, pasuje"      │
  │  4  │  Countering        │  Kontrpropozycja          │  "A może 19:00?"    │
  │  5  │  ProvidingDetails  │  Podanie danych           │  "Kowalski, 2 os."  │
  │  6  │  Confirming        │  KONIEC - Sukces ✅       │  "Dziękuję!"        │
  │  7  │  Failing           │  KONIEC - Porażka ❌      │  "Rozumiem, szkoda" │
  └────────────────────────────────────────────────────────────────────────────┘
```  

W kodzie używamy `Literal`, aby jednoznacznie oznaczyć każdą akcję. Niektóre z nich (jak `Asking`) niosą ze sobą konkretne dane ("payload"), które model musi uzupełnić.

In [3]:
class Greeting(BaseModel):
    """Przywitanie się"""
    action: Literal["greeting"]


class Asking(BaseModel):
    """Pytanie o dostępność stolika"""
    action: Literal["asking"]
    date: str
    time: str
    party_size: int


class Accepting(BaseModel):
    """Akceptacja propozycji restauracji"""
    action: Literal["accepting"]
    accepted_time: str


class Countering(BaseModel):
    """Kontrpropozycja gdy oferta poza zakresem"""
    action: Literal["countering"]
    proposed_time: str
    reason: str


class ProvidingDetails(BaseModel):
    """Podanie danych do rezerwacji"""
    action: Literal["providing_details"]
    customer_name: str
    special_requests: Optional[str] = None


class Confirming(BaseModel):
    """KONIEC - Sukces - potwierdzenie i pożegnanie"""
    action: Literal["confirming"]
    summary: str
    completed_steps: List[str]


class Failing(BaseModel):
    """KONIEC - Porażka - nie udało się zarezerwować"""
    action: Literal["failing"]
    reason: str
    completed_steps: List[str]

## `NextStep` - główny schemat decyzji agenta



To jest kluczowy moment, w którym łączymy teorię "Rolling Horizon Planning" z twardym kodem.

Struktura `NextStep` to implementacja mechanizmu, który można nazwać lejkiem decyzyjnym (*Decision Funnel*). Wymuszamy na modelu proces myślowy, który zawęża się od szerokiej analizy do jednego, konkretnego działania.

Zamiast prosić model o "rozmowę", prosimy go o wypełnienie struktury, która narzuca dyscyplinę:

1.  **Analiza (`current_state`)**: Najpierw model musi "głośno pomyśleć". Analizuje przeszłość (co powiedziała restauracja) i ocenia sytuację. To jest jego "brudnopis".
2.  **Strategia (`plan_remaining_steps`)**: Następnie model patrzy w przyszłość. Generuje listę kroków, które *wydaje mu się*, że trzeba będzie zrobić (np. `["Zapytaj o 19:30", "Podaj nazwisko"]`). Ważne: ten plan może się zmienić w kolejnej turze!
3.  **Taktyka (`next_action`)**: I tu dzieje się magia. Mimo że plan może mieć 5 kroków, model musi wybrać i wykonać **tylko ten jeden, pierwszy krok**.

To jest bezpiecznik przed halucynacjami. Model może marzyć o 5 krokach do przodu, ale fizycznie pozwalamy mu wykonać tylko jeden ruch, po którym znów wracamy do fazy obserwacji (pętla).

Oto kod spinający to w całość. Zwróć uwagę na pole `next_action` – używamy tu `Union`, aby wymusić wybór jednej z 7 zdefiniowanych wcześniej kart.

In [4]:
class NextStep(BaseModel):
    """Decyzja agenta - co myśli i co robi"""
    
    current_state: str = Field(
        description="Krótka analiza: co restauracja powiedziała?"
    )
    
    plan_remaining_steps: Annotated[List[str], MaxLen(5)] = Field(
        description="Pozostałe kroki do wykonania (pusty przy confirming/failing)"
    )
    
    next_action: Union[
        Greeting,
        Asking,
        Accepting,
        Countering,
        ProvidingDetails,
        Confirming,
        Failing,
    ] = Field(description="Wykonaj pierwszy krok z planu")

## `dispatch`
`
To jest moment, w którym "Ciało" wykonuje polecenie "Mózgu".

Funkcja `dispatch` pełni rolę tłumacza. Otrzymuje od modelu surowy, ustrukturyzowany obiekt (np. `AskAvailability` z konkretną datą) i zamienia go na naturalną wypowiedź w języku polskim (lub wywołanie API).

### ⚠️ Uwaga na marginesie
Poniższy kod to wersja "na rozgrzewkę". Widzisz tu proste `if`y (lub `match/case`) i zwykłe f-stringi. To wygląda trywialnie, ale na spokojnie. To przykład na rozgrzewkę :).

W trudniejszych systemach funkcja `dispatch` to Twoja pierwsza linia obrony. To tutaj umieszczasz "guardrails" i warunki stabilizujące.
- *Co jeśli model wygeneruje datę z przeszłości?* -> Dispatch to wyłapie.
- *Co jeśli `party_size` to 1000 osób?* -> Dispatch zablokuje wysłanie takiego zapytania.

Na razie jednak, dla przejrzystości, skupmy się na samej mechanice tłumaczenia obiektów na słowa:

In [5]:
def dispatch(action: BaseModel, goal: ReservationGoal) -> str:
    """Zamienia akcję agenta na wypowiedź tekstową."""
    
    if isinstance(action, Greeting):
        return "Dzień dobry, dzwonię w sprawie rezerwacji stolika."
    
    if isinstance(action, Asking):
        return (
            f"Czy mają Państwo wolny stolik na {action.date} "
            f"o godzinie {action.time} dla {action.party_size} osób?"
        )
    
    if isinstance(action, Accepting):
        return f"Tak, godzina {action.accepted_time} nam odpowiada."
    
    if isinstance(action, Countering):
        return f"{action.reason}. Czy byłaby możliwość o {action.proposed_time}?"
    
    if isinstance(action, ProvidingDetails):
        response = f"Na nazwisko {action.customer_name}."
        if action.special_requests:
            response += f" I jeśli można - {action.special_requests}."
        return response
    
    if isinstance(action, Confirming):
        return f"Świetnie, dziękuję bardzo. {action.summary}. Do widzenia!"
    
    if isinstance(action, Failing):
        return f"Rozumiem. {action.reason}. Dziękuję za informację, do widzenia."
    
    return "..."


def is_terminal(action: BaseModel) -> bool:
    """Sprawdza czy akcja kończy rozmowę."""
    return isinstance(action, (Confirming, Failing))

### Na czym polega ta elastyczność?

Wróćmy na chwilę do kodu z lekcji 1.3. Tam również używaliśmy LLM do zarządzania stanem, ale wyglądało to tak:

```python
# Prosty wybór sanu
class BookingStatus(str, Enum):
    START = "START"
    NEGOTIATION = "NEGOTIATION"
    FAILED = "FAILED"
    # ...
```


Tam model zwracał prosty string: `status="NEGOTIATION"`.
To był 100% *Black Box*. Jeśli model się pomylił (np. przerwał rozmowę, choć klient był chętny), nie masz pojęcia dlaczego. Widzisz tylko błędny wynik.

Teraz, dzięki strukturze `NextStep`, zamieniamy to na *Gray Box* (w teorii *White Box*, ale prawda jest taka, że nadal sporo nie wiemy). Model zwraca złożony obiekt, który daje nam wgląd w jego "procesor":

1.  **Analiza (`current_state`):** Widzisz tok rozumowania. Jeśli model podejmie dziwną decyzję, tutaj znajdziesz przyczynę (np. *"Uznałem 'chyba' za odmowę"*). To klucz do debugowania.
2.  **Plan (`plan_remaining_steps`):** Widzisz intencję na przyszłość, zanim zostanie wykonana.
3.  **Payload (`next_action`):** To najważniejsza zmiana. Stan nie jest już pustą etykietą. Jest obiektem (np. `Asking`), który niesie ze sobą konkretne dane niezbędne do wykonania akcji.


## System prompt

Nasza funkcja `build_prompt` nie jest zwykłym stringiem. To dynamiczny generator, który zamienia obiekt biznesowy (`ReservationGoal`) na instrukcję operacyjną dla modelu.

W architekturze Poziomu 5 prompt musi pełnić cztery konkretne funkcje. Zobacz, jak realizuje je ten kod:

1.  *Constraints*: Wstrzykujemy tu fakty, których agent nie może zignorować. Kluczowa jest tu linia o elastyczności: `(+/- {goal.time_flexibility_minutes} min)`. To jest granica autonomii. Agent "wie", że jeśli restauracja zaproponuje godzinę mieszczącą się w tym zakresie, ma prawo ją zaakceptować (stan `accepting`). Jeśli wykracza poza zakres – musi negocjować (`countering`) lub odmówić (`failing`).

2.  *Action Space:* Jawnie listujemy **"DOSTĘPNE AKCJE"**. Zwróć uwagę, że nazwy te (`greeting`, `asking` itd.) mapują się 1:1 na pola `Literal` w naszych klasach Pydantic. To "uziemia" model – wie on, że nie może wymyślić akcji "joke" albo "wait", bo nie ma ich w menu.

### ☝️ Pro-Tip Inżynierski:
*W tym przykładzie wpisaliśmy listę akcji "z palca" dla pełnej czytelności. Docelowo zwykle zakładamy, że liczba akcji może być zmienna (zawsze coś może się zmienić) — może ich być 3, 10 lub 15 — dlatego można generować tę listę automatycznie (jako fragment promptu). Dzięki temu kod jest jedynym źródłem prawdy, a prompt aktualizuje się sam po dodaniu nowej klasy.*

3.  *Reasoning Rules:* Sekcja **"ZASADY PLANOWANIA"** to nowość. Uczymy model, jak ma wypełniać pole `plan_remaining_steps`. Wyjaśniamy, że plan to kroki *przyszłe*, a przy zakończeniu rozmowy plan ma być pusty. To eliminuje halucynacje, gdzie model kończył rozmowę, ale w planie wciąż miał "pożegnanie".

4.  Wzorce zachowań (Heuristics): Zamiast pisać kod `if/else` dla każdego scenariusza, dajemy modelowi **wzorce przepływu** (Happy Path, Negocjacja, Porażka). `greeting → asking → providing_details → confirming`. To działa jak "mapa drogowa". Agent widzi te wzorce i próbuje dopasować aktualną sytuację do jednego z nich. Dzięki temu wie, że po `greeting` zazwyczaj następuje `asking`, a nie `confirming`.


In [6]:
def build_prompt(goal: ReservationGoal) -> str:
    """
    Generuje specyfikację misji dla agenta.
    Łączy cel biznesowy, dostępne narzędzia i reguły przejść między stanami.
    """
    return f"""Jesteś agentem rezerwującym stolik przez telefon.

CEL REZERWACJI:
- Restauracja: {goal.restaurant_name}
- Data: {goal.date}
- Godzina: {goal.preferred_time} (+/- {goal.time_flexibility_minutes} min)
- Liczba osób: {goal.party_size}
- Nazwisko: {goal.customer_name}
- Specjalne życzenia: {goal.special_requests or "brak"}

DOSTĘPNE AKCJE:
1. greeting         → Przywitanie się
2. asking           → Pytanie o dostępność stolika
3. accepting        → Akceptacja alternatywnej godziny (w zakresie elastyczności)
4. countering       → Kontrpropozycja (jeśli oferta poza zakresem)
5. providing_details→ Podanie nazwiska i życzeń
6. confirming       → KONIEC SUKCES ✅ (rezerwacja potwierdzona)
7. failing          → KONIEC PORAŻKA ❌ (brak możliwości)

ZASADY PLANOWANIA:
- plan_remaining_steps = kroki które ZOSTAŁY do zrobienia
- Przy confirming/failing → plan może być pusty []

TYPOWY PRZEPŁYW (happy path):
greeting → asking → providing_details → confirming

PRZEPŁYW Z NEGOCJACJĄ:
greeting → asking → accepting → providing_details → confirming

PRZEPŁYW Z PORAŻKĄ:
greeting → asking → countering → failing
"""

## `run_reservation_agent`

Funkcja `run_reservation_agent` zarządza całym cyklem życia rozmowy, od inicjalizacji po zakończenie. W tej funkcji spinamy w całość wszystko, co przygotowaliśmy wcześniej: System Prompt, Struktury Pydantic (`NextStep`) oraz funkcję `dispatch`.

Kluczowe mechanizmy w tym fragmencie:

1.  **Pamięć (robocza):** Lista `messages` rośnie z każdą turą. To dzięki niej Agent "pamięta", że przed chwilą odmówiono mu godziny 19:00 i nie pyta o nią ponownie.
2.  **Bezpiecznik:** Pętla `for i in range(10)` to twarde ograniczenie kosztów. Systemy autonomiczne potrafią wpaść w pętle nieskończone (np. ciągłe witanie się). Sztywny limit kroków to podstawa higieny na poziomie 5
3.  **Observability:** Zobacz, co drukujemy w konsoli. Nie tylko to, co agent *mówi* (`agent_says`), ale też to, co *myśli* (`current_state`) i co *planuje* (`plan_remaining_steps`). To zamienia "Black Box" w "Glass Box" – widzisz dokładnie, dlaczego podjął taką, a nie inną decyzję.
4.  **Symulacja świata:** W zmiennej `scenario_responses` symulujemy odpowiedzi restauracji. W środowisku produkcyjnym w tym miejscu byłoby wywołanie API do Speech-to-Text (Transkrypcji) lub obsługa websocketów.

In [7]:
def run_reservation_agent(goal: ReservationGoal, scenario_responses: List[str]):
    """Prowadzi rozmowę telefoniczną."""
    
    print(f"\n{'='*60}")
    print(f"📞 DZWONIĘ DO: {goal.restaurant_name}")
    print(f"   Cel: {goal.party_size} os., {goal.date} o {goal.preferred_time}")
    print('='*60)
    
    messages = [
        {"role": "system", "content": build_prompt(goal)},
        {"role": "user", "content": "[Restauracja odbiera telefon]"}
    ]
    
    response_idx = 0
    
    for i in range(10):
        step_name = f"step_{i+1}"
        print(f"\n--- {step_name} ---")
        
        # Wywołaj LLM
        try:
            step: NextStep = h.call_api(messages, NextStep)
        except Exception as e:
            print(f"⚠️ Błąd: {e}")
            break
        
        # Wykonaj dispatch
        agent_says = dispatch(step.next_action, goal)
        action_name = step.next_action.action.upper()
        
        print(f"🤖 [{action_name}]: {agent_says}")
        print(f"   Stan: {step.current_state}")
        print(f"   Plan: {step.plan_remaining_steps}")
        
        # Czy koniec rozmowy?
        if is_terminal(step.next_action):
            if isinstance(step.next_action, Confirming):
                print(f"\n✅ SUKCES!")
                print(f"   Wykonane kroki:")
                for s in step.next_action.completed_steps:
                    print(f"     - {s}")
            else:  # Failing
                print(f"\n❌ PORAŻKA: {step.next_action.reason}")
            return step
        
        # Dodaj do historii
        messages.append({"role": "assistant", "content": agent_says})
        
        # Odpowiedź restauracji
        if response_idx < len(scenario_responses):
            restaurant_reply = scenario_responses[response_idx]
            response_idx += 1
        else:
            restaurant_reply = "[Rozłączono]"
        
        print(f"👤 RESTAURACJA: {restaurant_reply}")
        messages.append({"role": "user", "content": restaurant_reply})
    
    print("\n⚠️ Przekroczono limit kroków")
    return None

## Sprawdźmy to w akcji!

```
┌───────────────────────────────────────────────────────────────────────────────────────┐
│                           MASZYNA STANÓW - 7 AKCJI                                    │
│                              (ReAct Agent)                                            │
├───────────────────────────────────────────────────────────────────────────────────────┤
│                                                                                       │
│                              ┌─────────────┐                                          │
│                              │   🚀 START  │                                          │
│                              └──────┬──────┘                                          │
│                                     │                                                 │
│                                     ▼                                                 │
│                          ┌───────────────────┐                                        │
│                          │   1. GREETING     │                                        │
│                          │  "Dzień dobry..." │                                        │
│                          └────────┬──────────┘                                        │
│                                   │                                                   │
│                                   ▼                                                   │
│                          ┌───────────────────┐                                        │
│                          │    2. ASKING      │                                        │
│                          │  "Czy macie...?"  │                                        │
│                          └────────┬──────────┘                                        │
│                                   │                                                   │
│                                   ▼                                                   │
│                    ┌──────────────────────────────┐                                   │
│                    │      ODPOWIEDŹ RESTAURACJI   │                                   │
│                    │         (OBSERVATION)        │                                   │
│                    └──────────────┬───────────────┘                                   │
│                                   │                                                   │
│         ┌─────────────────────────┼─────────────────────────┐                         │
│         │                         │                         │                         │
│         ▼                         ▼                         ▼                         │
│ ┌───────────────┐     ┌───────────────────┐     ┌───────────────────┐                 │
│ │  "OK, mamy    │     │ "Proponujemy inną │     │ "Brak miejsc"     │                 │
│ │   stolik"     │     │  godzinę: X"      │     │                   │                 │
│ └───────┬───────┘     └─────────┬─────────┘     └─────────┬─────────┘                 │
│         │                       │                         │                           │
│         │           ┌───────────┴───────────┐             │                           │
│         │           │                       │             │                           │
│         │           ▼                       ▼             │                           │
│         │   ┌───────────────┐     ┌─────────────────┐     │                           │
│         │   │ X w zakresie  │     │ X poza zakresem │     │                           │
│         │   │   (±30 min)   │     │                 │     │                           │
│         │   └───────┬───────┘     └────────┬────────┘     │                           │
│         │           │                      │              │                           │
│         │           ▼                      ▼              │                           │
│         │   ┌───────────────┐     ┌─────────────────┐     │                           │
│         │   │ 3. ACCEPTING  │     │  4. COUNTERING  │     │                           │
│         │   │ "Tak, X       │     │  "A może Y?"    │◀────┼──────────────┐            │
│         │   │  pasuje"      │     │                 │     │              │            │
│         │   └───────┬───────┘     └────────┬────────┘     │              │            │
│         │           │                      │              │              │            │
│         │           │                      ▼              │              │            │
│         │           │         ┌────────────────────────┐  │              │            │
│         │           │         │  ODPOWIEDŹ RESTAURACJI │  │              │            │
│         │           │         └───────────┬────────────┘  │              │            │
│         │           │                     │               │              │            │
│         │           │      ┌──────────────┼───────────┐   │              │            │
│         │           │      │              │           │   │              │            │
│         │           │      ▼              ▼           ▼   │              │            │
│         │           │  [akceptują]   [inna prop.]  [odm.] │              │            │
│         │           │      │              │           │   │              │            │
│         │           │      │              │           │   │     (jeśli limit          │
│         │           │      │              │           │   │     COUNTERING            │
│         │           │      │              │           │   │     nie osiągnięty)       │
│         │           │      │         ┌────┘           │   │              │            │
│         │           │      │         │                │   │              │            │
│         │           │      │    w zakresie?           │   │              │            │
│         │           │      │     ┌────┴────┐          │   │              │            │
│         │           │      │    TAK       NIE─────────┼───┼──────────────┘            │ 
│         │           │      │     │                    │   │                           │ 
│         │           │      │     ▼                    │   │                           │
│         │           │      │  ACCEPTING               │   │                           │
│         │           │      │     │                    │   │                           │
│         ▼           ▼      ▼     ▼                    ▼   ▼                           │
│    ┌──────────────────────────────────┐        ┌──────────────┐                       │
│    │      5. PROVIDING_DETAILS        │        │  7. FAILING  │                       │
│    │    "Na nazwisko Kowalski, 2 os." │        │  "Szkoda..." │                       │
│    └─────────────────┬────────────────┘        └──────┬───────┘                       │
│                      │                                │                               │
│                      ▼                                ▼                               │
│             ┌─────────────────┐              ┌────────────────┐                       │
│             │  6. CONFIRMING  │              │                │                       │
│             │  "Dziękuję!"    │              │   ❌ KONIEC    │                       │
│             └────────┬────────┘              │   (porażka)    │                       │
│                      │                       │                │                       │
│                      ▼                       └────────────────┘                       │
│             ┌────────────────┐                                                        │
│             │                │                                                        │
│             │   ✅ KONIEC    │                                                        │
│             │   (sukces)     │                                                        │
│             │                │                                                        │
│             └────────────────┘                                                        │
│                                                                                       │
├───────────────────────────────────────────────────────────────────────────────────────┤
│  PRZEPŁYWY:                                                                           │
│  ───────────────────────────────────────────────────────────────────────────────────  │
│  🟢 Happy Path:     GREETING → ASKING → PROVIDING_DETAILS → CONFIRMING ✅             │
│  🟡 Negocjacja OK:  GREETING → ASKING → ACCEPTING → PROVIDING_DETAILS → CONFIRMING ✅ │
│  🟠 Kontrpropozycja: GREETING → ASKING → COUNTERING → ACCEPTING → PROV... → CONF. ✅  │
│  🔴 Porażka:        GREETING → ASKING → [COUNTERING ×N] → FAILING ❌                  │
│                                                                                       │
├───────────────────────────────────────────────────────────────────────────────────────┤
│  STANY KOŃCOWE (TERMINAL):                                                            │
│  • CONFIRMING (6) → Sukces ✅   │  • FAILING (7) → Porażka ❌                         │
│                                                                                       │
│  GUARDRAILS:                                                                          │
│  • max_counter_proposals: 2 (limit negocjacji)                                        │
│  • time_flexibility: ±30 min (zakres akceptacji)                                      │
└───────────────────────────────────────────────────────────────────────────────────────┘
```

### SCENARIUSZ 1: `Happy path`

In [8]:
goal_1 = ReservationGoal(
    restaurant_name="Pizza Napoli",
    date="Jutro",
    preferred_time="19:00",
    party_size=2,
    customer_name="Kowalski"
)

responses_1 = [
    "Pizza Napoli, słucham?",
    "Tak, mamy wolny stolik o 19:00. Na jakie nazwisko?",
    "Kowalski, 2 osoby, 19:00. Zarezerwowane!"
]


run_reservation_agent(goal_1, responses_1)


📞 DZWONIĘ DO: Pizza Napoli
   Cel: 2 os., Jutro o 19:00

--- step_1 ---
🤖 [GREETING]: Dzień dobry, dzwonię w sprawie rezerwacji stolika.
   Stan: Połączenie zostało nawiązane, restauracja odebrała telefon. Czas rozpocząć rozmowę.
   Plan: ['asking', 'providing_details', 'confirming']
👤 RESTAURACJA: Pizza Napoli, słucham?

--- step_2 ---
🤖 [ASKING]: Czy mają Państwo wolny stolik na Jutro o godzinie 19:00 dla 2 osób?
   Stan: Restauracja zgłosiła się. Pora zapytać o dostępność stolika zgodnie z celem.
   Plan: ['providing_details', 'confirming']
👤 RESTAURACJA: Tak, mamy wolny stolik o 19:00. Na jakie nazwisko?

--- step_3 ---
🤖 [PROVIDING_DETAILS]: Na nazwisko Kowalski.
   Stan: Restauracja potwierdziła dostępność na 19:00 i pyta o nazwisko.
   Plan: ['confirming']
👤 RESTAURACJA: Kowalski, 2 osoby, 19:00. Zarezerwowane!

--- step_4 ---
🤖 [CONFIRMING]: Świetnie, dziękuję bardzo. Rezerwacja potwierdzona: Pizza Napoli, Jutro 19:00, 2 osoby, Kowalski.. Do widzenia!
   Stan: Restauracja potw

NextStep(current_state='Restauracja potwierdziła rezerwację (Kowalski, 2 osoby, 19:00). Zadanie zakończone sukcesem.', plan_remaining_steps=[], next_action=Confirming(action='confirming', summary='Rezerwacja potwierdzona: Pizza Napoli, Jutro 19:00, 2 osoby, Kowalski.', completed_steps=['greeting', 'asking', 'providing_details', 'confirming']))

## SCENARIUSZ 2: Negocjacja (alternatywa w zakresie)

In [9]:
goal_2 = ReservationGoal(
    restaurant_name="Bella Italia",
    date="Sobota",
    preferred_time="20:00",
    party_size=4,
    customer_name="Nowak",
    time_flexibility_minutes=60  # akceptuje 19:00-21:00
)
responses_2 = [
    "Bella Italia, w czym mogę pomóc?",
    "O 20:00 niestety zajęte. Mamy stolik o 20:30.",
    "Na jakie nazwisko?",
    "Nowak, 4 osoby, 20:30. Zapisane!"
]


run_reservation_agent(goal_2, responses_2)


📞 DZWONIĘ DO: Bella Italia
   Cel: 4 os., Sobota o 20:00

--- step_1 ---
🤖 [GREETING]: Dzień dobry, dzwonię w sprawie rezerwacji stolika.
   Stan: Restauracja odebrała telefon, rozpoczynam rozmowę.
   Plan: ['asking', 'providing_details', 'confirming']
👤 RESTAURACJA: Bella Italia, w czym mogę pomóc?

--- step_2 ---
🤖 [ASKING]: Czy mają Państwo wolny stolik na Sobota o godzinie 20:00 dla 4 osób?
   Stan: Restauracja odebrała telefon i pyta o cel kontaktu.
   Plan: ['providing_details', 'confirming']
👤 RESTAURACJA: O 20:00 niestety zajęte. Mamy stolik o 20:30.

--- step_3 ---
🤖 [ACCEPTING]: Tak, godzina 20:30 nam odpowiada.
   Stan: Restauracja zaproponowała 20:30, co mieści się w tolerancji +/- 60 min od 20:00.
   Plan: ['providing_details', 'confirming']
👤 RESTAURACJA: Na jakie nazwisko?

--- step_4 ---
🤖 [PROVIDING_DETAILS]: Na nazwisko Nowak.
   Stan: Restauracja zaakceptowała godzinę 20:30 i pyta o nazwisko
   Plan: ['confirming']
👤 RESTAURACJA: Nowak, 4 osoby, 20:30. Zapisane!

--

NextStep(current_state='Restauracja potwierdziła rezerwację na godzinę 20:30 dla 4 osób na nazwisko Nowak.', plan_remaining_steps=[], next_action=Confirming(action='confirming', summary='Rezerwacja potwierdzona: Sobota, 20:30, 4 osoby, Nowak.', completed_steps=['greeting', 'asking', 'accepting', 'providing_details', 'confirming']))

In [ ]:
## SCENARIUSZ 3: Porażka

In [10]:
goal_3 = ReservationGoal(
    restaurant_name="Pod Różą",
    date="Sylwester",
    preferred_time="20:00",
    party_size=6,
    customer_name="Wiśniewski"
)
responses_3 = [
    "Pod Różą, słucham.",
    "Niestety na Sylwestra wszystko zajęte. Cały wieczór."
]

run_reservation_agent(goal_3, responses_3)


📞 DZWONIĘ DO: Pod Różą
   Cel: 6 os., Sylwester o 20:00

--- step_1 ---
🤖 [GREETING]: Dzień dobry, dzwonię w sprawie rezerwacji stolika.
   Stan: Restauracja odebrała telefon. Muszę rozpocząć rozmowę.
   Plan: ['asking', 'providing_details', 'confirming']
👤 RESTAURACJA: Pod Różą, słucham.

--- step_2 ---
🤖 [ASKING]: Czy mają Państwo wolny stolik na Sylwester o godzinie 20:00 dla 6 osób?
   Stan: Restauracja potwierdziła odebranie i czeka na szczegóły.
   Plan: ['providing_details', 'confirming']
👤 RESTAURACJA: Niestety na Sylwestra wszystko zajęte. Cały wieczór.

--- step_3 ---
🤖 [FAILING]: Rozumiem. Brak wolnych stolików na cały wieczór sylwestrowy.. Dziękuję za informację, do widzenia.
   Stan: Restauracja poinformowała, że cały wieczór sylwestrowy jest zajęty, więc rezerwacja jest niemożliwa.
   Plan: []

❌ PORAŻKA: Brak wolnych stolików na cały wieczór sylwestrowy.


NextStep(current_state='Restauracja poinformowała, że cały wieczór sylwestrowy jest zajęty, więc rezerwacja jest niemożliwa.', plan_remaining_steps=[], next_action=Failing(action='failing', reason='Brak wolnych stolików na cały wieczór sylwestrowy.', completed_steps=['greeting', 'asking']))

## Budujemy pamięć agenta

Zanim pójdziemy dalej, musimy zrobić mały "krok w bok", który tak naprawdę jest skokiem w górę.

W Module 2 przerabialiśmy *Memory Engineering*. Mówiliśmy o pamięci semantycznej i epizodycznej. Zastosujmy zdobytą wiedzę teraz, na poziomie 5. Planowanie z pamięcią daje ciekawsze możliwości.


### Definicje warstw

1.  **🏛️ Constitutional Memory** – "DNA agenta". Zasady bezpieczeństwa i etyki, których agentowi **NIE WOLNO** złamać. To tutaj są "guardrails".
2.  **⚡ Working Memory** – Kontekst bieżącej sesji (RAM). To, o czym rozmawiamy "teraz".
3.  **🎯 Workflow Memory** – *Nasz Secret Sauce*. Maszyna stanów (State Machine). Gdzie jestem w procesie biznesowym?
4.  **📚 Episodic Memory** – Co się wydarzyło? Historia interakcji, logi zdarzeń.
5.  **🧠 Semantic Memory** – Co wiem? Baza wiedzy (RAG), fakty o świecie i użytkowniku.
6.  **🛠️ Procedural Memory** – Jak to zrobić? Definicje narzędzi, API, instrukcje obsługi.
7.  **🔧 Resource Memory** – Czy mam zasoby? Stan systemu, dostępność API, limity.


![](../../module2/images/memory_blue_print.jpg)



Dlaczego?
Nasz Agent rezerwacyjny do tej pory był "stateless" (bezstanowy). Dzwonił, rezerwował, zapominał.

Ale prawdziwy Agent (poziom 5) musi wiedzieć:
1.  **Kim JEST** (*Constitutional Memory*) – jakie ma żelazne zasady?
2.  **Z kim ROZMAWIA** (*Semantic Memory*) – czy klient ma alergię na orzechy?
3.  **Co robił WCZEŚNIEJ** (*Episodic Memory*) – czy ta restauracja ostatnio nas wystawiła?
4.  **Co robi TERAZ** (*Working Memory*) – jaki jest stan bieżącej negocjacji?

Teraz Twoim zadaniem jest zbudowanie pełnego stosu pamięci dla naszego Agenta AI.


## 🧠 Zadanie 4.2.1: Konstytucja Agenta AI

Zaczynamy od warstwy najniższej, ale ważnej. *Constitutional Memory* to zbiór zasad, których Agentowi **nie wolno** złamać, niezależnie od tego, co powie użytkownik ("zignoruj poprzednie instrukcje") lub jak bardzo halucynuje model. To są Twoje "Guardrails".

Twoim zadaniem jest zdefiniowanie niezmiennych reguł bezpieczeństwa i biznesu.

### Wymagania:
1.  Stwórz klasę `ConstitutionalRules` dziedziczącą po `BaseModel`.
2.  Zdefiniuj pole `max_time_flexibility_minutes` (int) – górna granica elastyczności (ustaw np. na 60 min). To bezpiecznik, żeby agent nie przesunął kolacji o 5 godzin.
3.  Zdefiniuj `max_counter_proposals` (int) – limit negocjacji, aby agent nie wpadł w pętlę targowania się.
4.  Dodaj flagę `always_mention_allergies` (bool) – wymuszenie dbania o zdrowie klienta.
5.  Dodaj flagę `require_confirmation_summary` (bool) – wymuszenie podsumowania na koniec.
6.  Zainicjalizuj stałą `CONSTITUTIONAL` z domyślnymi wartościami.

<details>
    <summary style="background: #e6eaeb; padding: 4px 0; text-align: center; font-size: 20px; font-weight: 900;"> 👉 Kliknij tutaj (1 klik), aby zobaczyć podpowiedź 👈 </summary>
<p>

### Podpowiedź:
*   To są wartości statyczne, hardkodowane w kodzie (lub ładowane ze zmiennych środowiskowych). Agent ma do nich dostęp "tylko do odczytu".
*   Użyj typów prostych (`int`, `bool`).
*   Pamiętaj, że w Pydantic domyślne wartości ustawiamy znakiem równości (np. `param: int = 10`).

<details>
    <summary style="background: #e6eaeb; padding: 4px 0; text-align: center; font-size: 20px; font-weight: 900;"> 👉 Kliknij tutaj (1 klik), aby zobaczyć odpowiedź 👈 </summary>
<p>

```python
# ═══════════════════════════════════════════════════════════════════════════
# 🏛️ CONSTITUTIONAL MEMORY - guardrails (niezmienne!)
# ═══════════════════════════════════════════════════════════════════════════

class ConstitutionalRules(BaseModel):
    """Zasady, których agent NIGDY nie złamie."""
    max_time_flexibility_minutes: int = 60  # Absolutne max, nawet jeśli klient chce więcej
    max_counter_proposals: int = 2          # Nie negocjuj w nieskończoność
    always_mention_allergies: bool = True   # Kwestia zdrowia!
    require_confirmation_summary: bool = True  # Zawsze powtórz szczegóły

CONSTITUTIONAL = ConstitutionalRules()
```

</p>
</details>
</p>
</details>


In [ ]:
## YOUR CODE HERE
class ConstitutionalRules(BaseModel):
    max_time_flexibility_minutes: int = 60
    max_counter_proposals: int = 2
    always_mention_allergies: bool = True
    require_confirmation_summary: bool = True

CONSTITUTIONAL = ConstitutionalRules()

## 🧠 Zadanie 4.2.2: Profil klienta (*Semantic Memory*) 

Agent musi znać swojego mocodawcę. To jest **Semantic Memory** – wiedza o świecie i użytkowniku, która jest trwała.

Twoim zadaniem jest stworzenie struktury, która pozwoli agentowi "personalizować" rezerwację. Jeśli Kowalski ma alergię na orzechy, agent musi to wiedzieć **zanim** zadzwoni, a nie pytać o to w trakcie rozmowy.

### Wymagania:
1.  Stwórz klasę `CustomerProfile` (BaseModel).
2.  Pola obowiązkowe: `email`, `name`.
3.  Pole `vip_level` (Optional[str]) – np. dla kluczowych klientów.
4.  Listy stringów: `allergies` (kluczowe dla bezpieczeństwa), `preferences` (np. "stolik przy oknie") oraz `rules` (specjalne instrukcje klienta).
5.  Stwórz przykładowy słownik `CUSTOMER_KNOWLEDGE`, zawierający profile dla "Jana Kowalskiego" (z alergią na orzechy) i "Anny Nowak" (rezerwacje służbowe).

<details>
    <summary style="background: #e6eaeb; padding: 4px 0; text-align: center; font-size: 20px; font-weight: 900;"> 👉 Kliknij tutaj (1 klik), aby zobaczyć podpowiedź 👈 </summary>
<p>

### Podpowiedź:
- Użyj `Optional` z modułu `typing` dla pól, które nie zawsze są dostępne.
- Pola listowe (np. `List[str]`) warto zainicjalizować pustą listą `[]`, aby uniknąć błędów `NoneType`.
- Zwróć uwagę, jak `rules` mogą wpływać na zachowanie agenta (np. "Pytać o fakturę").

<details>
    <summary style="background: #e6eaeb; padding: 4px 0; text-align: center; font-size: 20px; font-weight: 900;"> 👉 Kliknij tutaj (1 klik), aby zobaczyć odpowiedź 👈 </summary>
<p>

```python
class CustomerProfile(BaseModel):
    """Profil klienta z regułami i preferencjami."""
    email: str
    name: str
    vip_level: Optional[str] = None  # "gold", "platinum"
    allergies: List[str] = []
    preferences: List[str] = []
    rules: List[str] = []  # Specjalne reguły

# EXAMPLES how to use 
CUSTOMER_KNOWLEDGE: Dict[str, CustomerProfile] = {
    "kowalski@firma.pl": CustomerProfile(
        email="kowalski@firma.pl",
        name="Jan Kowalski",
        vip_level="gold",
        allergies=["orzechy"],
        preferences=["stolik przy oknie", "cicha okolica"],
        rules=[
            "Zawsze prosić o najlepszy stolik",
            "Informować o alergii na orzechy PRZED rezerwacją"
        ]
    ),
    "nowak@corp.com": CustomerProfile(
        email="nowak@corp.com",
        name="Anna Nowak",
        vip_level=None,
        allergies=[],
        preferences=["sala prywatna dla grup"],
        rules=["Rezerwacje służbowe - pytać o możliwość faktury"]
    )
}
```

</p>
</details>
</p>
</details>

In [14]:
from typing import Dict

## YOUR CODE HERE
class CustomerProfile(BaseModel):
    email: str
    name: str
    vip_level: Optional[str] = None  # "gold", "platinum"
    allergies: List[str] = []
    preferences: List[str] = []
    rules: List[str] = [] # specjalne reguły

CUSTOMER_KNOWLEDGE: Dict[str, CustomerProfile] = {
    "kowalski@firma.pl": CustomerProfile(
        email="kowalski@firma.pl",
        name="Jan Kowalski",
        vip_level="gold",
        allergies=["orzechy"],
        preferences=["stolik przy oknie", "cicha okolica"],
        rules=[
            "Zawsze prosić o najlepszy stolik",
            "Informować o alergii na orzechy PRZED rezerwacją"
        ]
    ),
    "nowak@corp.com": CustomerProfile(
        email="nowak@corp.com",
        name="Anna Nowak",
        vip_level=None,
        allergies=[],
        preferences=["sala prywatna dla grup"],
        rules=["Rezerwacje służbowe - pytać o możliwość faktury"]
    )
}

## 🧠 Zadanie 4.2.3: Uczenie się na błędach (*Episodic Memory*)

`"Ostatnim razem w 'Pod Różą' czekaliśmy 30 minut na stolik, mimo rezerwacji"`. To jest informacja, która zmienia strategię gry.

To jest **Episodic Memory**. Zbieramy historię interakcji, aby agent mógł wyciągać wnioski. Jeśli wiemy, że dana restauracja robi problemy, agent może być bardziej asertywny lub zaproponować inne miejsce.

### Wymagania:
1.  Stwórz klasę `PastReservation`.
2.  Zdefiniuj pola: `date`, `restaurant`, `outcome` (sukces/porażka/anulowane) oraz opcjonalne `notes`.
3.  Stwórz słownik `INTERACTION_HISTORY`, mapujący e-mail klienta na listę `PastReservation`.
4.  Dodaj wpis dla Kowalskiego, gdzie w jednej restauracji był sukces, a w innej ("Pod Różą") wystąpił problem z czasem oczekiwania.

<details>
    <summary style="background: #e6eaeb; padding: 4px 0; text-align: center; font-size: 20px; font-weight: 900;"> 👉 Kliknij tutaj (1 klik), aby zobaczyć podpowiedź 👈 </summary>
<p>

### Podpowiedź:
*   Episodic Memory w najprostszej formie to log zdarzeń. W systemach produkcyjnych trzymalibyśmy to w bazie wektorowej (np. Qdrant), ale tutaj wystarczy struktura JSON/Dict.
*   Kluczem jest pole `notes` lub `issues` – to z niego LLM będzie wyciągał wnioski ("unikać tego miejsca" lub "potwierdzić dwa razy").

<details>
    <summary style="background: #e6eaeb; padding: 4px 0; text-align: center; font-size: 20px; font-weight: 900;"> 👉 Kliknij tutaj (1 klik), aby zobaczyć odpowiedź 👈 </summary>
<p>

```python
class PastReservation(BaseModel):
    """Pojedyncza rezerwacja z przeszłości."""
    date: str
    restaurant: str
    outcome: str  # "success", "failed", "cancelled"
    notes: Optional[str] = None  # "czekał 20 min", "świetna obsługa"

INTERACTION_HISTORY: Dict[str, List[PastReservation]] = {
    "kowalski@firma.pl": [
        PastReservation(
            date="2024-01-15",
            restaurant="Bella Italia",
            outcome="success",
            notes="Stolik przy oknie, klient zadowolony"
        ),
        PastReservation(
            date="2024-02-20",
            restaurant="Pod Różą",
            outcome="success",
            notes="PROBLEM: czekał 30 min mimo rezerwacji"
        ),
    ]
}
```

</p>
</details>
</p>
</details>

In [15]:
## YOUR CODE HERE
class PastReservation(BaseModel):
    date: str
    restaurant: str
    outcome: str
    notes: Optional[str] = None

INTERACTION_HISTORY: Dict[str, List[PastReservation]] = {
    "kowalski@firma.pl": [
        PastReservation(
            date="2024-01-15",
            restaurant="Bella Italia",
            outcome="success",
            notes="Stolik przy oknie, klient zadowolony"
        ),
        PastReservation(
            date="2024-02-20",
            restaurant="Pod Różą",
            outcome="success",
            notes="PROBLEM: czekał 30 min mimo rezerwacji"
        ),
    ]
}


## 🧠 Zadanie 4.2.4: Wszystko w jednym miejscu (*Working Memory*)

Skoro mamy już zdefiniowane dane statyczne (Profil, Konstytucja), potrzebujemy obiektu, który będzie trzymał **dynamiczny stan sesji**.

Nie wrzucaj tu wszystkiego. To nie jest śmietnik. To jest RAM Twojego agenta. Trzymamy tu tylko to, co zmienia się w trakcie rozmowy.

Kluczową nowością na Poziomie 5 jest **`previous_plan`**. Musimy pamiętać, co planowaliśmy krok temu, aby zachować ciągłość myślenia (lub wykryć, że musimy zmienić strategię).

### Wymagania:
1.  Stwórz klasę `WorkingMemory`.
2.  **Historia:** Pole `messages` (lista słowników) – to standard.
3.  **Licznik negocjacji:** Pole `counter_proposals_count` (int) – musimy wiedzieć, ile razy już odmówiliśmy, żeby nie przekroczyć limitu z Konstytucji.
4.  **Ślad poznawczy:** Pole `previous_plan` (List[str]). To tutaj zapiszemy `plan_remaining_steps` z poprzedniej iteracji.
5.  **Domyślne wartości:** Upewnij się, że wszystkie pola mają bezpieczne wartości domyślne (puste listy, zera).

<details>
    <summary style="background: #e6eaeb; padding: 4px 0; text-align: center; font-size: 20px; font-weight: 900;"> 👉 Kliknij tutaj (1 klik), aby zobaczyć podpowiedź 👈 </summary>
<p>

### Podpowiedź:
*   Użyj `Field(default_factory=list)` dla list w Pydantic, aby uniknąć problemu współdzielenia tej samej listy między różnymi instancjami pamięci.
*   Nie dodawaj tu `ReservationGoal` ani `CustomerProfile`. One są przekazywane do funkcji `run_agent` jako argumenty (Read-Only Context), a nie są częścią mutowalnego stanu.

<details>
    <summary style="background: #e6eaeb; padding: 4px 0; text-align: center; font-size: 20px; font-weight: 900;"> 👉 Kliknij tutaj (1 klik), aby zobaczyć odpowiedź 👈 </summary>
<p>

```python
class WorkingMemory(BaseModel):
    """
    Dynamiczny stan sesji (RAM).
    Trzymamy tu tylko to, co ZMIENIA SIĘ w trakcie rozmowy.
    """
    # 1. Zapis rozmowy (Mutable History)
    messages: List[Dict[str, str]] = Field(default_factory=list)
    
    # 2. Liczniki sterujące (Control Flags)
    # Potrzebne, by przestrzegać Constitutional Memory (max 2 negocjacje)
    counter_proposals_count: int = 0
    
    # 3. Ślad myślowy (Cognitive Trace) - KLUCZOWE DLA LEVEL 5
    # Przechowujemy plan z POPRZEDNIEJ tury.
    # Dzięki temu wiemy, czy agent trzyma się kursu, czy właśnie zmienił zdanie.
    previous_plan: List[str] = Field(default_factory=list)
```

</p>
</details>
</p>
</details>

In [ ]:
## YOUR CODE HERE
class WorkingMemory(BaseModel):
    """
    Dynamiczny stan sesji (RAM).
    Trzymamy tu tylko to, co ZMIENIA SIĘ w trakcie rozmowy.
    """
    # 1. Zapis rozmowy (Mutable History)
    messages: List[Dict[str, str]] = Field(default_factory=list)
    
    # 2. Liczniki sterujące (Control Flags)
    # Potrzebne, by przestrzegać Constitutional Memory (max 2 negocjacje)
    counter_proposals_count: int = 0
    
    # 3. Ślad myślowy (Cognitive Trace) - KLUCZOWE DLA LEVEL 5
    # Przechowujemy plan z POPRZEDNIEJ tury.
    # Dzięki temu wiemy, czy agent trzyma się kursu, czy właśnie zmienił zdanie.
    previous_plan: List[str] = Field(default_factory=list)

## Podsumowanie 
Przeszliśmy drogę od liniowego myślenia do cyklicznego procesu decyzyjnego. Najważniejszy wniosek, z jakim chcę Cię zostawić, to zmiana paradygmatu: w Software 3.0 plan nie jest świętością, jest tylko hipotezą. Zamiast wymuszać na modelu realizację "listy zakupów", daliśmy mu narzędzia do oceny sytuacji (pętla ReAct) i zmiany kursu, gdy rzeczywistość (np. brak wolnego stolika) tego wymaga.

Z technicznego punktu widzenia na razie zamieniliśmy "Black Box" (gdzie model zwracał prosty string) na "Gray Box" (gdzie model zwraca obiekt `NextStep`). Dzięki temu widzimy nie tylko decyzję, ale też proces myślowy (*reasoning*), co drastycznie ułatwia debugowanie. Zbudowaliśmy też solidny system pamięci wielowarstwowej – od niezmiennej konstytucji po dynamiczną pamięć roboczą. To właśnie te struktury danych, a nie same prompty, stanowią o stabilności systemu agentowego.

To była solidna rozgrzewka. Na razie nasz mechanizm `dispatch()` jest prosty, a środowisko bezpieczne. W tej lekcji skupiliśmy się na "mechanizmie" połączenia kropek. W kolejnej lekcji będziemy kontynuować. Wykorzystasz te same wzorce projektowe – pętlę decyzyjną, separację pamięci i ustrukturyzowane wyjście – aby zbudować system do zamawiania taksówki. Tam dynamika będzie inna, ale fundament inżynierski pozostanie ten sam.